# Run Random

Run this notebook for the selected `DATASET` / `MODEL` combo.

This notebook:
- Runs the `random` baseline through the shared benchmark pipeline.
- Loads the matching checkpoint and explain index for the chosen combo.
- Saves official run artifacts under `resources/results/official_random/` and finishes with the shared one-spot metrics summary.

Usage:
- Change `DATASET` and `MODEL` in the first code cell when needed.
- Run the notebook top to bottom.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

_PROJECT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next((p for p in _PROJECT_CANDIDATES if (p / "I_explainer_benchmark").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root from the current working directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from I_explainer_benchmark.src.notebooks.explainer_notebook_setup import (
    initialize_explainer_notebook,
    prepare_explainer_run,
)

# Select the dataset / model combo here.
DATASET = "simulate_v1"
MODEL = "tgn"

# Shared notebook setup. Leave the rest unchanged.
BOOT = initialize_explainer_notebook("09_random.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)
NB = BOOT.env
CFG = BOOT.config
SETTINGS = BOOT.settings
BENCHMARK_ROOT = BOOT.bench_root
REPO_ROOT = BOOT.repo_root


In [2]:
import torch
from I_explainer_benchmark.src.notebooks.notebook_helpers import set_global_seed

MODEL = str(MODEL).strip().lower()
SUPPORTED_MODEL_TYPES = set(CFG["supported_model_types"])
if MODEL not in SUPPORTED_MODEL_TYPES:
    raise ValueError(f"Unsupported MODEL={MODEL!r}. Choose one of {sorted(SUPPORTED_MODEL_TYPES)}")

CTX = prepare_explainer_run("09_random.ipynb", dataset=DATASET, model=MODEL, start=PROJECT_ROOT)

DATASET_NAME = CTX.dataset
MODEL_TYPE = CTX.model
SEED = int(CFG["seed"])
set_global_seed(SEED)

N_EVAL_EVENTS = int(SETTINGS["n_eval_events"])
TARGET_EVENT_IDS = [int(v) for v in SETTINGS.get("target_event_ids", [])]
RUN_LABEL = str(SETTINGS.get("run_label", ""))
CANDIDATES_SIZE = int(SETTINGS["candidates_size"])
USE_CACHED_RANDOM_RESULTS = bool(SETTINGS["use_cached_random_results"])

EXPLAIN_INDEX_PATH = CTX.explain_index_path
CKPT_PATH = CTX.checkpoint_path
DEVICE = CTX.device
out_dir = BENCHMARK_ROOT / "resources" / "results" / str(CFG["out_dir_name"])
out_dir.mkdir(parents=True, exist_ok=True)

print("DATASET:", DATASET_NAME)
print("MODEL  :", MODEL_TYPE)
print("TARGETS:", TARGET_EVENT_IDS if TARGET_EVENT_IDS else f"first {N_EVAL_EVENTS} from explain index")
print("CKPT   :", CKPT_PATH)
print("DEVICE :", DEVICE)


DATASET: simulate_v1
MODEL  : tgn
TARGETS: first 30 from explain index
CKPT   : /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/models/simulate_v1/tgn/tgn_simulate_v1_best.pth
DEVICE : mps


In [3]:
# Run/reuse random explanations and persist results.jsonl for downstream metrics
import numpy as np
import pandas as pd

from I_explainer_benchmark.src.explainers.builder import make_explainer_builder
from I_explainer_benchmark.src.notebooks.explainer_notebook_runtime import (
    enrich_results_jsonl_with_candidates,
    load_jsonl_records,
    prepare_standard_edge_explainer_run,
    run_or_reuse_explanations,
)
from I_explainer_benchmark.src.notebooks.notebook_helpers import forced_target_event_ids_for_combo

FORCED_TARGET_EVENT_IDS = list(
    globals().get('FORCED_TARGET_EVENT_IDS', forced_target_event_ids_for_combo(DATASET_NAME, MODEL_TYPE))
)

build_explainer = make_explainer_builder(
    dataset_name=DATASET_NAME,
    model_type=MODEL_TYPE,
    device=DEVICE,
    seed=SEED,
    callable_scope=globals(),
)

random_explainer = build_explainer(
    'random',
    allow_missing=False,
    overrides={
        'alias': 'random',
        'seed': SEED,
        'reseed_each_call': True,
    },
)
print('Explainer alias:', random_explainer.alias)

ctx = prepare_standard_edge_explainer_run(
    dataset_name=DATASET_NAME,
    model_name=MODEL_TYPE,
    ckpt_path=CKPT_PATH,
    device=DEVICE,
    explain_index_path=EXPLAIN_INDEX_PATH,
    n_eval_events=N_EVAL_EVENTS,
    out_dir=out_dir,
    explainer=random_explainer,
    candidates_size=CANDIDATES_SIZE,
    seed=SEED,
    include_event_ids=FORCED_TARGET_EVENT_IDS,
    explicit_target_event_ids=TARGET_EVENT_IDS,
)
model = ctx.model
events = ctx.events
backbone = model.backbone
all_event_idxs = ctx.all_event_idxs
target_event_idxs = ctx.target_event_idxs
anchors = ctx.anchors

print('Anchors selected:', len(anchors), '/', len(all_event_idxs))
print('Target event_idxs:', target_event_idxs)
if anchors:
    print('First target event_idx:', anchors[0]['event_idx'])

out, run_dir, out_jsonl, base_name = run_or_reuse_explanations(
    runner=ctx.runner,
    anchors=anchors,
    dataset_name=DATASET_NAME,
    model_name=MODEL_TYPE,
    explainer_name='random',
    target_event_idxs=target_event_idxs,
    use_cached=USE_CACHED_RANDOM_RESULTS,
    run_label=RUN_LABEL,
)

updated_rows = enrich_results_jsonl_with_candidates(
    out_jsonl,
    extractor=ctx.extractor,
    events=events,
    dataset_name=DATASET_NAME,
    model=model,
)

print('run_dir:', run_dir)
print('out_jsonl:', out_jsonl)
print('base_name:', base_name)
print('Enriched JSONL rows:', updated_rows)

_records = load_jsonl_records(out_jsonl)

random_rows = []
for rec in _records:
    ctx_row = rec.get('context') or {}
    tgt = ctx_row.get('target') or {}
    res = rec.get('result') or {}
    extras = res.get('extras') or {}
    cand = list(extras.get('candidate_eidx') or [])
    imp = list(res.get('importance_edges') or [])
    random_rows.append({
        'event_idx': int(tgt.get('event_idx')) if tgt.get('event_idx') is not None else pd.NA,
        'candidate_size': int(len(cand)),
        'importance_size': int(len(imp)),
        'importance_mean': float(np.mean(np.asarray(imp, dtype=float))) if imp else np.nan,
        'importance_std': float(np.std(np.asarray(imp, dtype=float))) if imp else np.nan,
    })

random_results_df = pd.DataFrame(random_rows)
out_csv = out_dir / f'{base_name}.csv'
random_results_df.to_csv(out_csv, index=False)

print('Saved explanations:', out_csv)
print(random_results_df.head())


Built explainer 'random' from /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/configs/explainer/random.json
Explainer alias: random
102 events to explain
Anchors selected: 30 / 102
Target event_idxs: [3, 92, 142, 268, 386, 471, 711, 935, 1056, 1161, 1266, 1457, 1526, 1750, 1864, 1985, 2174, 2430, 2590, 2889, 3081, 3357, 3466, 3537, 3674, 3876, 4025, 4167, 4291, 4465]
First target event_idx: 3


Evaluation:   0%|          | 0/30 [00:00<?, ?expl/s]

Stored new random explanations under: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_random/simulate_v1_tgn_official_random_20260315_200656
run_dir: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_random/simulate_v1_tgn_official_random_20260315_200656
out_jsonl: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_random/simulate_v1_tgn_official_random_20260315_200656/results.jsonl
base_name: simulate_v1_tgn_official_random_20260315_200656
Enriched JSONL rows: 30
Saved explanations: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/official_random/simulate_v1_tgn_official_random_20260315_200656.csv
   event_idx  candidate_size  importance_size  importance_mean  i

In [4]:
# Shared metrics + export pipeline (clean notebook wrapper)
from I_explainer_benchmark.src.notebooks.notebook_metrics_pipeline import run_notebook_metrics_from_namespace

_pipeline_out = run_notebook_metrics_from_namespace(globals(), CFG)
globals().update(_pipeline_out)
run_dir_export = _pipeline_out['run_dir_export']
export_root = _pipeline_out['export_root']
out_jsonl = _pipeline_out['out_jsonl']
out_dir = _pipeline_out['out_dir']
base_name = _pipeline_out['base_name']
print('CURRENT_RUN_ID:', _pipeline_out['CURRENT_RUN_ID'])
print('Saved run export directory:', run_dir_export)
print('Updated global tables under:', export_root)


CURRENT_RUN_ID: simulate_v1_tgn_official_random_20260315_200656
Saved run export directory: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready/simulate_v1_tgn_official_random_20260315_200656
Updated global tables under: /Users/juliawenkmann/Documents/CodingProjects/master_thesis/time_to_explain_official/I_explainer_benchmark/resources/results/summary_ready


In [5]:
# One-spot metrics summary (official)
from I_explainer_benchmark.src.notebooks.notebook_display import show_one_spot_metrics_summary

_one_spot = globals().get("one_spot", globals().get("_pipeline_out", {}).get("one_spot", {}))
show_one_spot_metrics_summary(_one_spot)


One-spot metrics summary (official):


,dataset,model,explainer,variant,n_events,best_fid,best_fid_sparsity,aufsc,best_minus_aufsc,best_fid_raw,...,tempme_acc_auc.ratio_acc,tempme_acc_auc.ratio_prob,tempme_acc_auc.ratio_logit,tempme_acc_auc.ratio_aps,tempme_acc_auc.ratio_auc,temgx_aufsc,cody_AUFSC_plus,cody_AUFSC_minus,cody_CHARR,elapsed_sec
0,simulate_v1,tgn,random,official,30,1.6021053081,1.0,1.0085991988,0.5935061092,1.6021053081,...,0.8552083333,-0.0079507664,-0.081513562,0.8333333333,1.0,0.2239417132,0.0509066341,0.0045047814,0.0082771124,0.0000873389
